In [ ]:
# Part 1 — install, imports, config
# !pip install pdfplumber openpyxl pandas pytesseract -q
# !apt-get -qq install tesseract-ocr tesseract-ocr-hin     # Hindi OCR safety net

import pdfplumber, openpyxl, pandas as pd
import re, os, unicodedata, difflib, collections
from google.colab import files

CONFIG = dict(
    SPACE_GAP    = 0.25,   # gap (x font size) that counts as a space between English chars
    COLUMN_GAP   = 3.0,    # gap (x font size) that splits a line into columns (signatory block)
    USE_OCR      = True,   # Tesseract second opinion ONLY when the text-layer Hindi disagrees with Excel
    TESSDATA     = None,   # folder of hin.traineddata if not installed system-wide
    BAG_PASS_VOCAB = True, # PASS programme/discipline words whose glyphs match but PDF glyph ORDER differs (font encoding variant)
    DATE_HI = ["vkt chl twu nks gtkj NCchl dks laLFkku dh ;g",
               "eqnzk vafdr mikf/k iznku dh x;h"],          # change per convocation
    DATE_EN = "on this Twentieth day of June Two Thousand Twenty Six",
)

def norm(t): return re.sub(r'\s+',' ',str(t or '').strip())
def qn(s): return norm(re.sub(r"[’‘“”\"']","'",s))
def sim(a,b): return round(100*difflib.SequenceMatcher(None,str(a),str(b)).ratio(),1)
HI_FONT=re.compile(r'Ajay|Kruti|Devlys|Chanakya|Kokila|Mangal',re.I)


In [ ]:
# Part 2 — Kruti Dev → Unicode converter + unit tests
# ---- Kruti Dev -> Unicode ----
KD = {
 'vkS':'औ','vks':'ओ','vk':'आ','v':'अ','bZ':'ई','b':'इ','m':'उ','Å':'ऊ',',s':'ऐ',',':'ए','_':'ऋ',
 'd':'क','[k':'ख','x':'ग','?k':'घ','³':'ङ','p':'च','N':'छ','t':'ज','>k':'झ','¥':'ञ',
 'V':'ट','B':'ठ','M':'ड','<':'ढ','.k':'ण','r':'त','Fk':'थ','n':'द','/k':'ध','u':'न',
 'i':'प','Q':'फ','c':'ब','Hk':'भ','e':'म',';':'य','j':'र','y':'ल','˜':'ळ','o':'व',"'k":'श','"k':'ष',
 'l':'स','g':'ह','{k':'क्ष','K':'ज्ञ','=':'त्र','J':'श्र',
 'ç':'प्र','Ø':'क्र','Ü':'श्','æ':'द्र','ä':'क्त','Ô':'ह्य','#':'रु','$':'रू',')':'द्ध','|':'द्य','«':'त्त','Ý':'ङ्क','Þ':'ङ्ख',
 'D':'क्','[':'ख्','X':'ग्','?':'घ्','P':'च्','T':'ज्','>':'झ्','U':'न्','I':'प्','C':'ब्',
 'E':'म्','Y':'ल्','L':'स्','F':'थ्','H':'भ्','/':'ध्',"'":'श्','"':'ष्','.':'ण्','{':'क्ष्','¸':'य्','O':'व्','G':'ह्','R':'त्',
 'k':'ा','h':'ी','q':'ु','w':'ू','s':'े','S':'ै','a':'ं','¡':'ँ','%':'ः','~':'्','z':'्र','A':'।','f':'ि','W':'ृ',
 '¼':'(','½':')',
}
for c in ['[k','?k','>k','.k','/k',"'k",'"k','Fk','Hk','{k']:
    KD[c+'s']=KD[c]+'ो'; KD[c+'S']=KD[c]+'ौ'
_C=r'[क-हळ]़?'; _CL=rf'(?:{_C}्)*{_C}'; MAT='ािीुूृेैोौंँ'
def kd2uni(s, unmapped=None):
    s=(s or '').replace('\u00a0',' ')
    out=[];i=0
    while i<len(s):
        for L in (3,2,1):
            ch=s[i:i+L]
            if ch in KD: out.append(KD[ch]); i+=L; break
        else:
            c=s[i]
            if unmapped is not None and not c.isspace() and c not in 'Z.,-' : unmapped.add(c)
            out.append(c); i+=1
    t=''.join(out)
    t=t.replace('ाे','ो').replace('ाै','ौ')
    t=re.sub(rf'ि({_CL})',r'\1ि',t)                      # short-i after its cluster
    t=re.sub(rf'({_CL})([{MAT}]*)Z',r'र्\1\2',t)         # reph before cluster
    t=t.replace('Z','र्')
    t=t.replace('ा्र','्रा')                              # rakar must follow consonant, not matra
    t=t.replace('प्राद्यैो','प्रौद्यो')                       # known font variant: ै glyph emitted after the next conjunct
    t=t.replace('िा','ि')                                 # font draws an extra bar glyph after wide conjunct+i
    return t
def hn(s):
    s=unicodedata.normalize('NFC',str(s or '')); s=s.replace('\u200c','').replace('\u200d','')
    return re.sub(r'\s+',' ',s.strip())
def bag(s):                                  # order-insensitive glyph bag (matras split like Kruti types them)
    s=hn(s).replace('ो','ाे').replace('ौ','ाै').replace('\u093c','')
    return ''.join(sorted(re.sub(r'\s+','',s)))
def soft(s):
    s=unicodedata.normalize('NFD',hn(s)).replace('\u093c','').replace('ँ','ं')
    return re.sub(r'\s+','',unicodedata.normalize('NFC',s))

# --- unit tests (must all print OK) ---
_T={'vfu#) Jhjke':'अनिरुद्ध श्रीराम','vkfÜou çlkn':'आश्विन प्रसाद','fllksfn;k /kzqoh gsesaæflag':'सिसोदिया ध्रुवी हेमेंद्रसिंह',
    "pOgk.k f'koe~ lq/kkdj lqo.kkZ":'चव्हाण शिवम् सुधाकर सुवर्णा','lekdfyr lfdZV fMtkbu ,oa çkS|ksfxdh':'समाकलित सर्किट डिजाइन एवं प्रौद्योगिकी',
    'jklk;fud vfHk;k¡f=kdh':'रासायनिक अभियाँत्रिकी','laLFkku dk vf/kLukrd fMIyksek ¼ihthMhvkbZvkbZVh½':'संस्थान का अधिस्नातक डिप्लोमा (पीजीडीआईआईटी)',
    'dh mikf/k çca/ku esa ekbuj ds lkFk çnku djrk gS':'की उपाधि प्रबंधन में माइनर के साथ प्रदान करता है'}
for k,v in _T.items(): print('OK ' if hn(kd2uni(k))==hn(v) else 'BAD',k,'->',kd2uni(k))


In [ ]:
# Part 3 — load Excel + Hindi vocabulary from the "Hindi Degree Name" sheet
# ---------- DB ----------
HAS_SR={'MA','MSc','MTech','PhD','e-Masters'}
DEG_KEYS=['Bachelor','Master','Doctor','Diploma','Executive','Three']
def key(s): return re.sub(r'\s+',' ',str(s or '').lower().replace('minors','minor')).strip()
def load_db(path):
    wb=openpyxl.load_workbook(path,data_only=True); db={}; conflicts=[]
    for sn in wb.sheetnames:
        if sn=='Hindi Degree Name': continue
        off=1 if sn in HAS_SR else 0
        for row in wb[sn].iter_rows(min_row=2,values_only=True):
            try: roll=str(int(float(str(row[off]))))
            except Exception: continue
            deg=''
            for c in row[off+3:]:
                if c and isinstance(c,str) and any(k in c for k in DEG_KEYS): deg=norm(c)
            rec=dict(name=norm(row[off+1]),name_hi=hn(row[off+2]),degree=deg,sheet=sn,programme=norm(row[off+3]))
            if roll in db and sn!='EG_Phase 0' and db[roll]['sheet']!='EG_Phase 0': conflicts.append((roll,db[roll]['sheet'],sn))
            if roll not in db or sn!='EG_Phase 0':
                if roll in db and db[roll]['sheet']=='EG_Phase 0' and (db[roll]['name'],db[roll]['degree'],db[roll]['name_hi'])!=(rec['name'],rec['degree'],rec['name_hi']):
                    rec['eg_diff']=db[roll]
                db[roll]=rec
    return db,load_vocab(wb),conflicts
def load_vocab(wb):
    V=dict(prog={},disc={},minor={},hon={})
    mode='deg'
    for r in wb['Hindi Degree Name'].iter_rows(min_row=2,values_only=True):
        r=[norm(c) if c is not None else '' for c in r[:6]]
        if not any(r): continue
        if r[0]=='Minor in English': mode='minor'; continue
        if mode=='deg':
            full,disc,hp,hd=r[1],r[2],r[4],r[5]
            if full and hp: V['prog'][key(full)]=hp
            if disc and hd:
                for a,b in zip(re.split(r'\s*/\s*',disc),re.split(r'\s*/\s*',hd)): V['disc'][key(a)]=b
        else:
            if r[0] and r[1]: V['minor'][key(r[0])]=r[1]
            if r[2] and r[3]: V['hon'][key(r[2])]=r[3]
    return V
DEG_RE=re.compile(r'^(?P<prog>(?:Three\s*-\s*Year\s+)?(?:Bachelor of \w+|Master of \w+|Executive Masters|Doctor of Philosophy|Post Graduate Diploma of the Institute))(?P<hon>\s+with Honours)?(?:\s+in\s+(?P<disc>.+?))?(?P<minor>\s+with Minors?\s+in\s+.+)?$')
def parse_degree(d):
    parts=re.split(r'\s+and\s+(?=(?:Bachelor|Master|Executive|Doctor|Post Graduate)\b)',norm(d)); out=[]
    for p in parts:
        m=DEG_RE.match(p); out.append(m.groupdict() if m else dict(prog=p,hon=None,disc=None,minor=None))
    return out


In [ ]:
# Part 4 — PDF → lines (stream order, Hindi/English by font)
# ---------- PDF -> lines ----------
def _fix_spaces(cs):
    cs=list(cs); i=0; w=lambda c:c['x1']-c['x0']
    while i<len(cs)-1:
        if cs[i]['text']==' ' and w(cs[i+1])<1.2 and cs[i+1]['text']!=' ':
            j=i+1
            while j<len(cs) and w(cs[j])<1.2 and cs[j]['text']!=' ': j+=1
            cs[i:j]=cs[i+1:j]+[cs[i]]; i=j
        else: i+=1
    return cs
def pdf_lines(page):
    rows=[]
    for c in page.chars:                                  # content-stream order kept
        ym=(c['top']+c['bottom'])/2
        for r in rows:
            if abs(ym-r['y'])<=max(3.0,0.4*c['size']): r['c'].append(c); break
        else: rows.append({'y':ym,'c':[c]})
    lines=[]
    for r in sorted(rows,key=lambda r:r['y']):
        xs=sorted(r['c'],key=lambda c:c['x0']); cuts=[]
        for p,c in zip(xs,xs[1:]):
            if c['x0']-p['x1']>CONFIG['COLUMN_GAP']*max(p['size'],c['size']): cuts.append((p['x1']+c['x0'])/2)
        segs=collections.defaultdict(list)
        for c in r['c']: segs[sum(c['x0']>t for t in cuts)].append(c)
        for k in sorted(segs):
            sg=segs[k]; hindi=any(HI_FONT.search(c['fontname']) for c in sg)
            if hindi: txt=''.join(c['text'] for c in _fix_spaces(sg))
            else:
                sx=sorted(sg,key=lambda c:c['x0']); txt=''
                for j,c in enumerate(sx):
                    if j and c['text']!=' ' and not txt.endswith(' ') and c['x0']-sx[j-1]['x1']>CONFIG['SPACE_GAP']*c['size']: txt+=' '
                    txt+=c['text']
            txt=norm(txt)
            if txt: lines.append(dict(text=txt,hindi=hindi,x0=min(c['x0'] for c in sg),x1=max(c['x1'] for c in sg),top=min(c['top'] for c in sg),bottom=max(c['bottom'] for c in sg),fonts=sorted({c['fontname'].split('+')[-1] for c in sg})))
    return lines


In [ ]:
# Part 5 — template slots + verification (PASS / REVIEW / FAIL)
# ---------- template ----------
G=dict(sen='vfHk"kn~ dh vuq\'kalk ij',ko='dks',men='esa',and_=',oa',plain='dh mikf/k iznku djrk gS',pg='iznku djrk gS')
EN_FIXED=['On the recommendation of the Senate','hereby confers on','Given under the seal of the Institute']
SIGN=["v/;{k] vfHk'kn~",'dqylfpo',"v/;{k] 'kklh e.My",'Chairman, Senate','Registrar','Chairman, Board of Governors']
def parse_page(L):
    T=[qn(l['text']) for l in L]
    def idx(g,s=0): return next((i for i in range(s,len(T)) if T[i]==qn(g)),None)
    P=dict(i_sen=idx(G['sen'])); P['i_ko']=idx(G['ko'],P['i_sen'] or 0)
    P['i_d1']=idx(CONFIG['DATE_HI'][0]); P['i_d2']=idx(CONFIG['DATE_HI'][1])
    P['i_conf']=P['i_d1']-1 if P['i_d1'] else None
    P['i_rec']=idx(EN_FIXED[0]); P['i_hc']=idx(EN_FIXED[1]); P['i_seal']=idx(EN_FIXED[2]); P['i_den']=idx(CONFIG['DATE_EN'],P['i_seal'] or 0)
    g=lambda a,b:[i for i in range(a+1,b)] if a is not None and b is not None and b>a else []
    P['name_hi_i']=g(P['i_sen'],P['i_ko']); P['block_i']=g(P['i_ko'],P['i_conf']); P['en_i']=g(P['i_hc'],P['i_seal'])
    P['T']=T; return P


# ---------- compare ----------
def cmp_en(exp,found):
    if exp==found: return 'PASS'
    return 'REVIEW' if exp.casefold()==found.casefold() else 'FAIL'
def ocr_lines(page,lns):
    """OCR the given line boxes with Tesseract (Hindi). Returns None if OCR is unavailable."""
    try:
        import pytesseract
        cfg='--psm 7'+(f" --tessdata-dir {CONFIG['TESSDATA']}" if CONFIG['TESSDATA'] else '')
        out=[]
        for ln in lns:
            box=(max(0,ln['x0']-6),max(0,ln['top']-6),min(page.width,ln['x1']+6),min(page.height,ln['bottom']+6))
            img=page.crop(box).to_image(resolution=300).original
            out.append(' '.join(pytesseract.image_to_string(img,lang='hin',config=cfg).split()))
        return ' '.join(out)
    except Exception:
        return None

def cmp_hi(exp,raw,vocab=False,ocr=None):
    st,conv,note=_cmp_hi_core(exp,raw,vocab)
    if st=='FAIL' and ocr is not None and CONFIG['USE_OCR']:
        o=ocr()
        if o is not None:
            if hn(o)==hn(exp): return 'REVIEW',conv,(note+' | ' if note else '')+'OCR matches Excel -> text-layer converter probably lacks a glyph (send me this raw text)'
            return 'FAIL',conv,(note+' | ' if note else '')+f'OCR also differs: {hn(o)!r}'
    return st,conv,note

def _cmp_hi_core(exp,raw,vocab=False):
    um=set(); conv=kd2uni(raw,um); e,f=hn(exp),hn(conv)
    if e==f: return 'PASS',conv,''
    if um: return 'FAIL',conv,f'unmapped glyph(s): {sorted(um)}'
    if bag(e)==bag(f): return ('PASS' if vocab and CONFIG['BAG_PASS_VOCAB'] else 'REVIEW'),conv,'same glyphs, PDF glyph order differs (font encoding variant)'
    if soft(e)==soft(f): return 'REVIEW',conv,'only space/anusvara/chandrabindu/nukta differs'
    return 'FAIL',conv,f'similarity {sim(e,f)}%'

def verify_page(page,db,V,n_pages=1):
    L=pdf_lines(page); P=parse_page(L); T=P['T']; res=[]
    add=lambda f,s,e='',fd='',n='':res.append((f,s,e,fd,n))
    m=re.search(r'Roll\s*No\.?\s*(\d{6,})',' '.join(T)); roll=m.group(1) if m else None
    if n_pages!=1: add('Pages','REVIEW','1',str(n_pages))
    if roll not in db: add('Roll','FAIL','present in Excel',roll or 'not found on PDF','Roll not in DB'); return roll,res
    s=db[roll]; add('Roll','PASS',roll,roll)
    if 'eg_diff' in s: add('DB consistency','REVIEW','sheets agree','EG_Phase 0 differs from '+s['sheet'],str(s['eg_diff']))
    def fx(label,i,g): add(label,'PASS' if i is not None else 'FAIL',g,L[i]['text'] if i is not None else 'MISSING')
    fx('Fixed HI: senate recommendation',P['i_sen'],G['sen']); fx('Fixed HI: ko',P['i_ko'],G['ko'])
    fx('Fixed HI: date line 1',P['i_d1'],CONFIG['DATE_HI'][0]); fx('Fixed HI: date line 2',P['i_d2'],CONFIG['DATE_HI'][1])
    for lab,i,g in (('rec',P['i_rec'],EN_FIXED[0]),('confers',P['i_hc'],EN_FIXED[1]),('seal',P['i_seal'],EN_FIXED[2]),('date',P['i_den'],CONFIG['DATE_EN'])): fx('Fixed EN: '+lab,i,g)
    for t in SIGN: ok=any(qn(t) in x for x in T); add('Signatory: '+t,'PASS' if ok else 'FAIL',t,'present' if ok else 'MISSING')

    parts=parse_degree(s['degree']); dual=len(parts)>1; pg=parts[0]['prog'].startswith('Post Graduate')
    hon=any(p['hon'] for p in parts); minor=norm(parts[0]['minor']) if not dual and parts[0]['minor'] else ''

    # English variable part
    en=[L[i]['text'] for i in P['en_i']]; name_en=en[0] if en else ''; rest=en[1:]
    lead=rest[0] if rest and rest[0] in ('The','The Degree of') else ''; deg_txt=norm(' '.join(rest[1:] if lead else rest))
    add('Name (EN)',cmp_en(s['name'],norm(name_en)),s['name'],norm(name_en))
    add('Degree intro (EN)','PASS' if lead==('The' if pg else 'The Degree of') else 'FAIL','The' if pg else 'The Degree of',lead or 'MISSING')
    add('Degree (EN)',cmp_en(s['degree'],deg_txt),s['degree'],deg_txt)

    # Hindi name
    nraw=' '.join(L[i]['text'] for i in P['name_hi_i']); st,cv,nt=cmp_hi(s['name_hi'],nraw,ocr=lambda:ocr_lines(page,[L[i] for i in P['name_hi_i']])); add('Name (HI)',st,s['name_hi'],cv,(nt+' ' if nt else '')+f'raw={nraw!r}')

    # Hindi degree block
    block=[L[i] for i in P['block_i']]; seps=[qn(l['text']) for l in block if qn(l['text']) in (G['men'],G['and_'])]
    content=[l['text'] for l in block if qn(l['text']) not in (G['men'],G['and_'])]; content_l=[l for l in block if qn(l['text']) not in (G['men'],G['and_'])]
    def ref(p):
        d=V['disc'].get(key(p['disc'])) if p['disc'] else None; g=V['prog'].get(key(re.sub(r'^Three\s*-\s*Year\s+','',p['prog'])))
        return d,g
    if hon or (dual and (parts[0]['minor'] or parts[1]['minor'])):
        add('Degree block (HI)','REVIEW','(honours / dual+minor layout not yet templated)',' | '.join(kd2uni(x) for x in content),'send a sample certificate of this type')
    elif dual:
        add('Fixed HI: and (,oa)','PASS' if seps==[G['and_']] else 'FAIL',G['and_'],str(seps))
        for k,p in enumerate(parts):
            d,g=ref(p); lab=f'Degree line (HI) {k+1}'
            if k>=len(content): add(lab,'FAIL','line present','MISSING'); continue
            if d and g: st,cv,nt=cmp_hi(f'{d} में {g}',content[k],vocab=True,ocr=lambda k=k:ocr_lines(page,[content_l[k]])); add(lab,st,f'{d} में {g}',cv,nt)
            else: add(lab,'REVIEW','no Hindi reference in sheet for '+str(p['disc'] or p['prog']),kd2uni(content[k]))
    else:
        add('Fixed HI: men (esa)','PASS' if seps==[G['men']] else 'FAIL',G['men'],str(seps))
        d,g=ref(parts[0])
        if len(content)!=2: add('Discipline (HI)','FAIL','2 lines (discipline, degree type)',f'{len(content)} lines')
        else:
            for lab,e,raw,what,li in (('Discipline (HI)',d,content[0],parts[0]['disc'],0),('Degree type (HI)',g,content[1],parts[0]['prog'],1)):
                if e: st,cv,nt=cmp_hi(e,raw,vocab=True,ocr=lambda li=li:ocr_lines(page,[content_l[li]])); add(lab,st,e,cv,nt)
                else: add(lab,'REVIEW','no Hindi reference in sheet for '+str(what),kd2uni(raw))

    # confers line
    ci=P['i_conf']; craw=L[ci]['text'] if ci is not None else None
    if craw is None: add('Hindi confers line','FAIL','present','MISSING')
    elif minor and not hon and not dual:
        mh=V['minor'].get(key(minor))
        if mh: st,cv,nt=cmp_hi(f'की उपाधि {mh} प्रदान करता है',craw,vocab=True,ocr=lambda:ocr_lines(page,[L[ci]])); add('Hindi confers line (minor)',st,f'की उपाधि {mh} प्रदान करता है',cv,nt)
        else: add('Hindi confers line (minor)','REVIEW','no Hindi reference for '+minor,kd2uni(craw))
    elif hon or dual and parts[0]['minor']: add('Hindi confers line','REVIEW','(not yet templated)',kd2uni(craw))
    else:
        g=G['pg'] if pg else G['plain']; add('Hindi confers line','PASS' if qn(craw)==qn(g) else 'FAIL',g,craw)

    used=set(P['name_hi_i']+P['block_i']+P['en_i'])|{P[k] for k in ('i_sen','i_ko','i_conf','i_d1','i_d2','i_rec','i_hc','i_seal','i_den') if P[k] is not None}
    used|={i for i,t in enumerate(T) if 'Roll No' in t or any(qn(x) in t for x in SIGN)}
    un=[L[i]['text'] for i in range(len(L)) if i not in used]
    add('Unexpected lines','PASS' if not un else 'REVIEW','none','; '.join(un))
    return roll,res
def overall(res):
    st=[r[1] for r in res]; return 'FAIL' if 'FAIL' in st else ('REVIEW' if 'REVIEW' in st else 'PASS')


In [ ]:
# Part 6 — debug helper + run() (upload cells unchanged)
FILL={'PASS':'C6EFCE','FAIL':'FFC7CE','REVIEW':'FFEB9C'}
RANK={'PASS':0,'REVIEW':1,'FAIL':2}
GROUPS={'Name (EN)':('Name (EN)',),'Degree (EN)':('Degree (EN)','Degree intro'),
        'Name (HI)':('Name (HI)',),'Degree/Discipline (HI)':('Discipline (HI)','Degree type (HI)','Degree line (HI)','Degree block (HI)','Hindi confers','Fixed HI: men','Fixed HI: and')}
def worst(sts): return max(sts,key=lambda s:RANK[s]) if sts else '-'

def debug_pdf(path):
    with pdfplumber.open(path) as pdf:
        for ln in pdf_lines(pdf.pages[0]):
            print(f"{ln['top']:6.1f} | {ln['fonts']} | {ln['text']!r}"+(f"  ->  {kd2uni(ln['text'])}" if ln['hindi'] else ''))

def run():
    print("📄 Upload the Excel database file:")
    excel = list(files.upload().keys())[0]
    print("\n📁 Upload PDF certificate(s):")
    pdfs = [f for f in files.upload().keys() if f.lower().endswith('.pdf')]

    db, V, conflicts = load_db(excel)
    print(f"\n✅ Loaded {len(db)} students from Excel | Hindi vocabulary: {({k:len(v) for k,v in V.items()})}")
    if conflicts: print(f"⚠️ {len(conflicts)} roll numbers appear in more than one category sheet, e.g. {conflicts[:3]}")

    summary, detail = [], []
    for pdf in sorted(pdfs):
        try:
            with pdfplumber.open(pdf) as doc:
                roll, res = verify_page(doc.pages[0], db, V, n_pages=len(doc.pages))
            st = overall(res)
        except Exception as e:                      # one bad PDF must not stop the batch
            roll, res, st = '?', [('ERROR','FAIL','','',repr(e))], 'FAIL'
        row = {'File': os.path.basename(pdf), 'Roll': roll or '?', 'Status': st}
        for g, prefixes in GROUPS.items():
            row[g] = worst([r[1] for r in res if r[0].startswith(prefixes)])
        row['Problems'] = '; '.join(f"{r[0]}: expected {r[2]!r} found {r[3]!r} {r[4] if r[1]!='PASS' else ''}".strip() for r in res if r[1]!='PASS')[:500]
        summary.append(row)
        for f,s,e,fd,note in res:
            detail.append({'File':os.path.basename(pdf),'Roll':roll,'Field':f,'Status':s,'Expected':e,'Found':fd,'Similarity %':sim(e,fd),'Note':note})

    df, dd = pd.DataFrame(summary), pd.DataFrame(detail)
    print(df.drop(columns=['Problems']).to_string(index=False))
    gaps = dd[dd['Note'].str.contains('unmapped glyph', na=False)]
    if len(gaps):
        print("\n🔧 Glyphs missing from the converter (send me these lines):")
        for _, g in gaps.iterrows(): print('  ', g['Roll'], g['Field'], '|', g['Note'])
    c = df['Status'].value_counts().to_dict()
    print(f"\n{'='*50}\nTotal: {len(df)} | ✅ PASS: {c.get('PASS',0)} | 🟡 REVIEW: {c.get('REVIEW',0)} | ❌ FAIL: {c.get('FAIL',0)}")

    out = 'verification_report.xlsx'
    with pd.ExcelWriter(out, engine='openpyxl') as w:
        df.to_excel(w, sheet_name='Summary', index=False)
        dd.to_excel(w, sheet_name='Field details', index=False)
        from openpyxl.styles import PatternFill
        for ws in w.book.worksheets:
            for r in ws.iter_rows(min_row=2):
                for cell in r:
                    if cell.value in FILL: cell.fill = PatternFill('solid', fgColor=FILL[cell.value])
            for col in ws.columns:
                ws.column_dimensions[col[0].column_letter].width = min(60, max(12, max(len(str(c.value or '')) for c in col)+2))
    files.download(out)
    return df, dd

results, details = run()


In [ ]:
# Part 7 — EXTRACT details from certificates (no Excel database needed)
# Needs Parts 1-5 to have been run first. Upload cell is the same as before.

def extract_certificate(page):
    """Return the personal details printed on one certificate page (English + Hindi)."""
    L = pdf_lines(page); P = parse_page(L); T = P['T']
    m = re.search(r'Roll\s*No\.?\s*(\d{6,})', ' '.join(T))
    # ---- English ----
    en = [L[i]['text'] for i in P['en_i']]
    name_en = en[0] if en else ''
    rest = en[1:]
    if rest and rest[0] in ('The', 'The Degree of'): rest = rest[1:]
    degree_full = norm(' '.join(rest))
    parts = parse_degree(degree_full)
    deg_en  = ' | '.join(norm(p['prog'] + (p['hon'] or '')) for p in parts)
    disc_en = ' | '.join(norm(p['disc'] or '') for p in parts)
    mino_en = ' | '.join(norm(p['minor'] or '') for p in parts if p['minor'])
    # ---- Hindi ----
    name_hi = kd2uni(' '.join(L[i]['text'] for i in P['name_hi_i']))
    block = [L[i]['text'] for i in P['block_i']]
    content = [t for t in block if qn(t) not in (G['men'], G['and_'])]
    dual = any(qn(t) == G['and_'] for t in block)
    disc_hi, deg_hi = [], []
    if dual:                                       # each line = "<discipline> में <degree>"
        for t in content:
            d, _, g = kd2uni(t).partition(' में ')
            disc_hi.append(d.strip()); deg_hi.append(g.strip())
    elif len(content) >= 2:                        # line 1 = discipline, line 2 = degree
        disc_hi.append(kd2uni(content[0])); deg_hi.append(kd2uni(content[1]))
    elif len(content) == 1:                        # e.g. PhD: only a degree line
        deg_hi.append(kd2uni(content[0]))
    conf = kd2uni(L[P['i_conf']]['text']) if P['i_conf'] is not None else ''
    mino_hi = re.sub(r'^की उपाधि\s*|\s*प्रदान करता है$', '', conf).strip()
    if mino_hi == conf and conf in ('प्रदान करता है',): mino_hi = ''
    return {
        'Roll No': m.group(1) if m else '',
        'Name (English)': name_en,        'Name (Hindi)': name_hi,
        'Degree (English)': deg_en,       'Degree (Hindi)': ' | '.join(deg_hi),
        'Discipline (English)': disc_en,  'Discipline (Hindi)': ' | '.join(disc_hi),
        'Minor/Honours (English)': mino_en, 'Minor/Honours (Hindi)': mino_hi,
        'Full degree text (English)': degree_full,
    }

def extract_all():
    print("📁 Upload PDF certificate(s):")
    pdfs = [f for f in files.upload().keys() if f.lower().endswith('.pdf')]
    rows = []
    for pdf in sorted(pdfs):
        try:
            with pdfplumber.open(pdf) as doc:
                for pg_no, page in enumerate(doc.pages, 1):     # also works for merged PDFs
                    r = extract_certificate(page); r = {'File': os.path.basename(pdf), 'Page': pg_no, **r}
                    rows.append(r)
        except Exception as e:                                  # one bad PDF must not stop the batch
            rows.append({'File': os.path.basename(pdf), 'Page': '-', 'Roll No': 'ERROR: ' + repr(e)})
    df = pd.DataFrame(rows)

    for _, r in df.iterrows():                                  # readable print-out
        print('-' * 70)
        for k in ['File', 'Roll No', 'Name (English)', 'Name (Hindi)', 'Degree (English)', 'Degree (Hindi)',
                  'Discipline (English)', 'Discipline (Hindi)', 'Minor/Honours (English)', 'Minor/Honours (Hindi)']:
            if str(r.get(k, '')).strip(): print(f"{k:26}: {r[k]}")

    out = 'extracted_certificates.xlsx'
    with pd.ExcelWriter(out, engine='openpyxl') as w:
        df.to_excel(w, sheet_name='Extracted', index=False)
        ws = w.book['Extracted']
        for col in ws.columns:
            ws.column_dimensions[col[0].column_letter].width = min(60, max(12, max(len(str(c.value or '')) for c in col) + 2))
    files.download(out)
    return df

extracted = extract_all()
